<a href="https://colab.research.google.com/github/E-HAZMATs/FT-SmolLM/blob/main/FT_SmolLM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
! pip -qqq install transformers datasets trl torch
! pip -qqq install peft

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 889.0/889.0 kB 17.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 31.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 19.9 MB/s eta 0:00:00


In [2]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from trl import SFTTrainer, SFTConfig
from datasets import load_dataset

In [3]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
device

'cuda'

In [4]:
dataset_name = 'HuggingFaceTB/smoltalk2'
ds = load_dataset(dataset_name, 'SFT', streaming=True)

README.md:   0%|          | 0.00/23.5k [00:00<?, ?B/s]

Resolving data files:   0%|          | 0/124 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/113 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/113 [00:00<?, ?it/s]

In [5]:
ds.keys()

dict_keys(['LongAlign_64k_Qwen3_32B_yarn_131k_think', 'OpenThoughts3_1.2M_think', 'aya_dataset_Qwen3_32B_think', 'multi_turn_reasoning_if_think', 's1k_1.1_think', 'smolagents_toolcalling_traces_think', 'smoltalk_everyday_convs_reasoning_Qwen3_32B_think', 'smoltalk_multilingual8_Qwen3_32B_think', 'smoltalk_systemchats_Qwen3_32B_think', 'table_gpt_Qwen3_32B_think', 'LongAlign_64k_context_lang_annotated_lang_6_no_think', 'Mixture_of_Thoughts_science_no_think', 'OpenHermes_2.5_no_think', 'OpenThoughts3_1.2M_no_think_no_think', 'hermes_function_calling_v1_no_think', 'smoltalk_multilingual_8languages_lang_5_no_think', 'smoltalk_smollm3_everyday_conversations_no_think', 'smoltalk_smollm3_explore_instruct_rewriting_no_think', 'smoltalk_smollm3_smol_magpie_ultra_no_think', 'smoltalk_smollm3_smol_rewrite_no_think', 'smoltalk_smollm3_smol_summarize_no_think', 'smoltalk_smollm3_systemchats_30k_no_think', 'table_gpt_no_think', 'tulu_3_sft_personas_instruction_following_no_think', 'xlam_traces_no_th

In [6]:
!find ~/.cache/huggingface/hub -iname "*SmolLM-135M*" -type d

In [7]:
!cat ~/.cache/huggingface/hub/models--HuggingFaceTB--SmolLM-135M/snapshots/*/config.json

cat: '/root/.cache/huggingface/hub/models--HuggingFaceTB--SmolLM-135M/snapshots/*/config.json': No such file or directory


In [8]:
split = 'Mixture_of_Thoughts_science_no_think'

In [9]:
model_name = "HuggingFaceTB/SmolLM-135M"
base = AutoModelForCausalLM.from_pretrained(model_name).to(device)
tokenizer = AutoTokenizer.from_pretrained(model_name + '-Instruct')

config.json:   0%|          | 0.00/724 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  538MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/272 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/111 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/3.59k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/801k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/565 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.10M [00:00<?, ?B/s]

In [10]:
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = 'right'

In [11]:
instucut_name = model_name + '-Instruct'
instruct = AutoModelForCausalLM.from_pretrained(instucut_name).to(device)


model.safetensors: reconstructing file:   0%|          |  0.00B /  269MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/272 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/156 [00:00<?, ?B/s]

In [12]:
print(f"Model Size: {(base.get_memory_footprint() / 1024**3):.3f}GB")

Model Size: 0.251GB


In [13]:
from datasets import Dataset
examples_list = list(ds[split].take(5))
examples = Dataset.from_list(examples_list)

In [14]:
tokenizer.chat_template

"{% for message in messages %}{{'<|im_start|>' + message['role'] + '\n' + message['content'] + '<|im_end|>' + '\n'}}{% endfor %}{% if add_generation_prompt %}{{ '<|im_start|>assistant\n' }}{% endif %}"

In [15]:
def format_ds(batch):
    return {'text': [tokenizer.apply_chat_template(ex, tokenize=False) for ex in batch['messages']]}

text = examples.map(format_ds, batched=True, remove_columns=examples.column_names)

Map:   0%|          | 0/5 [00:00<?, ? examples/s]

In [16]:
text[0]

{'text': "<|im_start|>user\nWhat hormone's action is inhibited by caffeine, leading to increased urination?A: ADH\nB: Insulin\nC: Thyroxine\nD: Cortisol<|im_end|>\n<|im_start|>assistant\nCaffeine acts as a diuretic by inhibiting the action of antidiuretic hormone (ADH), which is responsible for signaling the kidneys to reabsorb water and concentrate urine. When ADH is suppressed, the kidneys excrete more water, leading to increased urination. The other hormones listed—insulin (regulates blood sugar), thyroxine (regulates metabolism), and cortisol (involved in stress response)—are not directly related to fluid balance or diuresis. \n\n**Answer: A**  \n\\boxed{A}<|im_end|>\n"}

In [17]:
tokenizer.eos_token_id

2

In [18]:
# prompt = "I'd like a recipe for something warm."
# tokenized = tokenizer(prompt, return_tensors='pt').to(device)

# with torch.no_grad():
#   outputs = base.generate(
#       **tokenized,
#       max_new_tokens=200,
#       temperature=0.7,
#       do_sample=True,
#       pad_token_id=tokenizer.eos_token_id
#   )
#   decoded = tokenizer.decode(outputs[0], skip_special_tokens=True)
#   print(decoded)



In [19]:
torch.cuda.is_available()

True

In [20]:
base.device

device(type='cuda', index=0)

In [21]:
instruct.name_or_path

'HuggingFaceTB/SmolLM-135M-Instruct'

In [22]:
# # Sampling from the instruct model

# message = [{
#     "role": "user",
#     "content": prompt
# }]

# formatted = tokenizer.apply_chat_template(message, tokenize=False, add_generation_prompt=True)

# formatted_tokenized = tokenizer(formatted, return_tensors='pt').to(device)

# output = instruct.generate(
#     **formatted_tokenized,
#     max_new_tokens=200,
#     temperature=0.5,
#     do_sample=True,
#     pad_token_id=tokenizer.eos_token_id
# )

# decoded = tokenizer.decode(output[0])
# print(decoded)
# #

In [23]:
# assistant_start = tokenizer.bos_token + 'assistant\n'
# response_start = decoded.find(assistant_start) + len(assistant_start)
# response = decoded[response_start:].split(tokenizer.eos_token)[0]
# print(response)

### Now FT the base model

In [24]:
# Store checkpoints inside my google drive.
from google.colab import drive

drive.mount('/content/drive')

Mounted at /content/drive


In [25]:
def preprocess(batch):
  return {'text': [tokenizer.apply_chat_template(ex, tokenize=False) for ex in batch['messages']]}


In [26]:
output_dir=f"/content/drive/MyDrive/checkpoints/{model_name}-Mine"

training_config = SFTConfig(
    # Model and data
    output_dir=output_dir,
    dataset_text_field="text",
    max_length=512,

    # Training hyperparameters
    per_device_train_batch_size=160,  # Tried mutliple batchsizes for a few iters. 160 batch size uses 12.8/15 GB.
    gradient_accumulation_steps=2,
    learning_rate=5e-5,
    num_train_epochs=1,  # Start with 1 epoch
    # Need to estimate how many tokens the splits i'm using have to pick a good iter number.
    # Considering sequence packing, it's hard to get an accurate computation of how many batches the dataset fits.
    max_steps=2000,

    # Optimization
    warmup_steps=50,
    weight_decay=0.01,
    optim="adamw_torch",

    # Logging and saving
    logging_steps=10,
    eval_strategy="steps",
    save_strategy="steps",
    save_steps=35, # GPU disconnects at ~60 iters -_-
    eval_steps=35,
    load_best_model_at_end=True,
    metric_for_best_model='eval_loss',
    greater_is_better=False,
    save_total_limit=2,

    # Memory optimization
    dataloader_num_workers=0,
    # group_by_length=True,  # Group similar length sequences

    # Hugging Face Hub integration
    push_to_hub=False,  # Set to True to upload to Hub
    hub_model_id=f"your-username/{model_name}",

    # Experiment tracking
    # report_to=["trackio"],  # Use trackio for experiment tracking
    run_name=f"{model_name}-training",
)

print("Training configuration set!")
print(f"Effective batch size: {training_config.per_device_train_batch_size * training_config.gradient_accumulation_steps}")

Training configuration set!
Effective batch size: 320


In [27]:
used_splits = [
    'smoltalk_smollm3_everyday_conversations_no_think','Mixture_of_Thoughts_science_no_think',
    'tulu_3_sft_personas_instruction_following_no_think',
    'smoltalk_multilingual_8languages_lang_5_no_think',
    'table_gpt_no_think'
    ]

# Print a row from each split to check format.
for split in used_splits:
    example = next(iter(ds[split].take(1)))
    print(f"--- {split} ---")
    print(example)
    print()


--- smoltalk_smollm3_everyday_conversations_no_think ---
{'messages': [{'content': 'Hi there', 'role': 'user'}, {'content': 'Hello! How can I help you today?', 'role': 'assistant'}, {'content': "I'm looking for a healthy breakfast idea. What's a good option?", 'role': 'user'}, {'content': "A fruit salad is a great choice. It's nutritious, delicious, and easy to make.", 'role': 'assistant'}, {'content': 'What fruits are good in a fruit salad?', 'role': 'user'}, {'content': 'You can use a mix of your favorite fruits, such as strawberries, bananas, grapes, and pineapple.', 'role': 'assistant'}], 'chat_template_kwargs': {'custom_instructions': '', 'enable_thinking': False, 'python_tools': [], 'xml_tools': []}, 'source': 'smoltalk-smollm3_everyday-conversations'}

--- Mixture_of_Thoughts_science_no_think ---
{'messages': [{'content': "What hormone's action is inhibited by caffeine, leading to increased urination?A: ADH\nB: Insulin\nC: Thyroxine\nD: Cortisol", 'role': 'user'}, {'content': 'C

In [28]:
base.config

LlamaConfig {
  "architectures": [
    "LlamaForCausalLM"
  ],
  "attention_bias": false,
  "attention_dropout": 0.0,
  "bos_token_id": 0,
  "dtype": "bfloat16",
  "eos_token_id": 0,
  "head_dim": 64,
  "hidden_act": "silu",
  "hidden_size": 576,
  "initializer_range": 0.02,
  "intermediate_size": 1536,
  "max_position_embeddings": 2048,
  "mlp_bias": false,
  "model_type": "llama",
  "num_attention_heads": 9,
  "num_hidden_layers": 30,
  "num_key_value_heads": 3,
  "pad_token_id": null,
  "pretraining_tp": 1,
  "rms_norm_eps": 1e-05,
  "rope_parameters": {
    "rope_theta": 10000.0,
    "rope_type": "default"
  },
  "tie_word_embeddings": true,
  "transformers_version": "5.13.1",
  "use_cache": true,
  "vocab_size": 49152
}

In [29]:
from datasets import load_dataset_builder
row_count = 0
split_sizes = []
builder = load_dataset_builder("HuggingFaceTB/smoltalk2", "SFT")
for split in used_splits:
  count = builder.info.splits[split].num_examples
  split_sizes.append(count)
  row_count += count
  print(f"{split}: {count}")

print(f"Total rows: {row_count}")

Resolving data files:   0%|          | 0/124 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/113 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/113 [00:00<?, ?it/s]

smoltalk_smollm3_everyday_conversations_no_think: 2260
Mixture_of_Thoughts_science_no_think: 86110
tulu_3_sft_personas_instruction_following_no_think: 29970
smoltalk_multilingual_8languages_lang_5_no_think: 254047
table_gpt_no_think: 13203
Total rows: 385590


In [30]:
split_sizes

[2260, 86110, 29970, 254047, 13203]

In [31]:
from datasets import interleave_datasets
eval_ex_num = 400
eval_ds = []
train_ds = []
for split in used_splits:
  s = ds[split].map(preprocess, batched=True, remove_columns=ds[split].column_names)
  eval_ds.append(list(s.take(eval_ex_num)))
  train_ds.append(s.skip(eval_ex_num))

eval_ds = [ex for part in eval_ds for ex in part] # Flatten the eval list of lists
probas = [(size - eval_ex_num) / (row_count - eval_ex_num * len(used_splits)) for size in split_sizes]

## interleave_ds here sorta orders the train ds by selection from each split based on the probas.
# This way you get more diverse examples from the start instead of exhausting each split at a time.
train_ds = interleave_datasets(train_ds, probabilities=probas, stopping_strategy="all_exhausted")


In [32]:
x = next(iter(ds[used_splits[0]]))
x

{'messages': [{'content': 'Hi there', 'role': 'user'},
  {'content': 'Hello! How can I help you today?', 'role': 'assistant'},
  {'content': "I'm looking for a healthy breakfast idea. What's a good option?",
   'role': 'user'},
  {'content': "A fruit salad is a great choice. It's nutritious, delicious, and easy to make.",
   'role': 'assistant'},
  {'content': 'What fruits are good in a fruit salad?', 'role': 'user'},
  {'content': 'You can use a mix of your favorite fruits, such as strawberries, bananas, grapes, and pineapple.',
   'role': 'assistant'}],
 'chat_template_kwargs': {'custom_instructions': '',
  'enable_thinking': False,
  'python_tools': [],
  'xml_tools': []},
 'source': 'smoltalk-smollm3_everyday-conversations'}

In [33]:
# Checking how many big one sequence is.
# Does TRL handle if a tokenized row produces more tokens than sequence_len?
tokenized_example = tokenizer.apply_chat_template(x['messages'], tokenize=True)
tokenized_example['input_ids'].__len__()

108

In [34]:
eval_ds[0]

{'text': "<|im_start|>user\nHi there<|im_end|>\n<|im_start|>assistant\nHello! How can I help you today?<|im_end|>\n<|im_start|>user\nI'm looking for a healthy breakfast idea. What's a good option?<|im_end|>\n<|im_start|>assistant\nA fruit salad is a great choice. It's nutritious, delicious, and easy to make.<|im_end|>\n<|im_start|>user\nWhat fruits are good in a fruit salad?<|im_end|>\n<|im_start|>assistant\nYou can use a mix of your favorite fruits, such as strawberries, bananas, grapes, and pineapple.<|im_end|>\n"}

In [35]:
from datasets import Dataset
eval_ds = Dataset.from_list(eval_ds)

In [36]:

trainer = SFTTrainer(
    model=base,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    args=training_config,
    processing_class=tokenizer,
)

Adding EOS to eval dataset:   0%|          | 0/2000 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/2000 [00:00<?, ? examples/s]

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (2993 > 2048). Running this sequence through the model will result in indexing errors


Building labels for eval dataset:   0%|          | 0/2000 [00:00<?, ? examples/s]

Truncating eval dataset:   0%|          | 0/2000 [00:00<?, ? examples/s]

Dropping fully masked examples from eval dataset:   0%|          | 0/2000 [00:00<?, ? examples/s]

In [ ]:
trainer.train()

Step,Training Loss,Validation Loss


In [ ]:
10 iters in 14m30s